# Few-Shot Fine-tuning + RAG over 100 Diseases (Synthetic)

- **Fine-tuning (few-shot):** LoRA on only 10 diseases, 4 examples each. Its job is to learn the 5-section answer structure.
- **RAG:** a knowledge base of 100 diseases, one paragraph each.
- **Goal:** Fine-tuned + RAG must answer in the required format for all 100 diseases, including the 90 it never trained on.

Runtime: T4 GPU. Expected run time is about 15 minutes.
All data is synthetic and for learning only. It is not medical advice.

In [1]:
!pip -q install transformers peft accelerate sentence-transformers pandas python-docx

In [2]:
import time, random
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model
from sentence_transformers import SentenceTransformer

random.seed(0); torch.manual_seed(0)
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

device: cuda


## 1. Data: 100 diseases

Each disease has five fields. The RAG paragraph and the training answer are both built from these same fields, so the facts stay consistent across the pipeline.

In [3]:
DISEASES = [
 ("Type 2 Diabetes","increased thirst, frequent urination, fatigue, blurred vision","insulin resistance, excess body weight, physical inactivity","metformin (generic), dose set by a doctor","balanced diet, regular exercise, blood sugar monitoring"),
 ("Hypertension","often no symptoms, headache, dizziness, nosebleeds in severe cases","high salt intake, obesity, ageing, chronic stress","amlodipine or losartan (generic), dose set by a doctor","low-salt diet, regular exercise, limiting alcohol, regular blood pressure checks"),
 ("Asthma","wheezing, shortness of breath, chest tightness, night-time cough","allergens, respiratory infections, air pollution, genetic predisposition","salbutamol reliever and inhaled corticosteroid controller","avoiding triggers, asthma action plan, correct inhaler technique"),
 ("Migraine","throbbing one-sided headache, nausea, light sensitivity","hormonal changes, poor sleep, stress, certain foods","sumatriptan (generic) for attacks, dose set by a doctor","rest in a dark room, regular sleep, hydration, trigger diary"),
 ("Urinary Tract Infection","burning urination, frequent urge to urinate, cloudy urine","bacteria such as E. coli entering the urinary tract","nitrofurantoin or trimethoprim (generic) course, as prescribed","completing the full course, drinking plenty of water"),
 ("Iron-Deficiency Anemia","tiredness, pale skin, dizziness, brittle nails","low iron intake, heavy menstrual bleeding, blood loss","oral ferrous sulfate (generic), dose set by a doctor","iron-rich foods, vitamin C with meals, follow-up blood tests"),
 ("Dengue Fever","high fever, severe headache, pain behind the eyes, rash","dengue virus transmitted by Aedes mosquito bites","paracetamol for fever; avoid aspirin and ibuprofen","rest, oral fluids, platelet monitoring, urgent care for warning signs"),
 ("GERD","heartburn, sour taste, regurgitation, chest discomfort after meals","weak lower oesophageal sphincter, obesity, large meals","omeprazole (generic), dose set by a doctor","smaller meals, not lying down after eating, raising the head of the bed"),
 ("Hypothyroidism","weight gain, constipation, cold intolerance, dry skin, fatigue","Hashimoto thyroiditis, iodine deficiency, thyroid surgery","levothyroxine (generic), dose set by a doctor","daily dose on an empty stomach, periodic TSH blood tests"),
 ("Scabies","intense itching worse at night, rash with small bumps or burrow tracks","mite Sarcoptes scabiei spread by prolonged skin contact","permethrin 5% cream (generic), as directed","treating the whole body, washing bedding, treating close contacts"),
 ("Pneumonia","cough with phlegm, fever, chest pain on breathing, breathlessness","Streptococcus pneumoniae and other bacteria, viruses","amoxicillin (generic) for mild cases, as prescribed","rest, fluids, vaccination against pneumococcus and influenza"),
 ("Tuberculosis","cough lasting over three weeks, night sweats, weight loss, fever","Mycobacterium tuberculosis spread by airborne droplets","isoniazid, rifampicin, pyrazinamide and ethambutol combination, as prescribed","completing about six months of treatment, contact tracing"),
 ("Malaria","cyclical fever, chills, sweating, headache, muscle aches","Plasmodium parasites transmitted by infected Anopheles mosquitoes","artemisinin-based combination therapy, as prescribed","bed nets, insecticide spraying, prophylaxis for travellers"),
 ("Typhoid Fever","sustained fever, abdominal pain, headache, constipation or diarrhoea","Salmonella Typhi from contaminated food and water","azithromycin or ceftriaxone, as prescribed","safe drinking water, handwashing, typhoid vaccination"),
 ("Chickenpox","mild fever, itchy blister rash that crusts over","varicella-zoster virus spread by droplets and contact","paracetamol for fever, antihistamines for itch","calamine lotion, short nails, varicella vaccination"),
 ("Influenza","sudden fever, cough, sore throat, body aches, fatigue","influenza A and B viruses spread by droplets","oseltamivir (generic) if started early, as prescribed","rest, fluids, annual vaccination, hand hygiene"),
 ("Hepatitis B","jaundice, dark urine, fatigue, abdominal pain","hepatitis B virus spread by blood, sex and mother to child","tenofovir or entecavir for chronic infection, as prescribed","hepatitis B vaccination, liver function monitoring"),
 ("Osteoarthritis","joint pain worsening with activity, morning stiffness, reduced movement","cartilage wear with ageing, obesity, joint injury","paracetamol or topical NSAID gel, as directed","weight loss, physiotherapy, low-impact exercise"),
 ("Rheumatoid Arthritis","symmetric swollen painful joints, morning stiffness","autoimmune reaction in the joint lining, genetics, smoking","methotrexate (generic) as a disease-modifying drug","early specialist review, physiotherapy, biologics when needed"),
 ("Gout","sudden severe pain and swelling of the big toe, redness","uric acid crystal buildup, high purine diet, alcohol","colchicine or NSAIDs for attacks, allopurinol for prevention","hydration, limiting alcohol and sugary drinks, diet changes"),
 ("Coronary Artery Disease","chest pressure on exertion, breathlessness, fatigue","plaque buildup in heart arteries, high cholesterol, smoking","atorvastatin (generic) and aspirin, as prescribed","exercise, healthy diet, quitting smoking, cardiac rehabilitation"),
 ("Stroke","sudden face droop, arm weakness, slurred speech, confusion","blocked or burst blood vessel in the brain","clot-dissolving drugs or antiplatelets, as prescribed","urgent emergency care, blood pressure control, rehabilitation"),
 ("Chronic Kidney Disease","swelling in legs, fatigue, reduced urine, itching","diabetes, hypertension, glomerulonephritis","ACE inhibitor or ARB (generic) to protect kidneys","blood sugar and blood pressure control, low-salt diet, regular monitoring"),
 ("Epilepsy","recurrent seizures, staring spells, brief loss of awareness","genetic factors, brain injury, stroke, infection","levetiracetam or carbamazepine (generic), dose set by a specialist","consistent medication, avoiding sleep deprivation, driving restrictions"),
 ("Depression","persistent low mood, loss of interest, sleep and appetite changes, fatigue","biological, psychological and social factors, life stress","SSRI antidepressant such as sertraline (generic), as prescribed","cognitive behavioural therapy, exercise, social support"),
 ("Glaucoma","gradual loss of peripheral vision, often without pain","raised eye pressure damaging the optic nerve, family history","prostaglandin analogue or beta-blocker eye drops","regular eye checks, laser treatment or surgery if drops fail"),
 ("Cataract","blurred vision, glare, faded colours, night-time difficulty","ageing, diabetes, eye injury, long-term steroid use","no drug treatment; surgery when vision is affected","stronger glasses early, lens replacement surgery"),
 ("Psoriasis","thick red scaly patches on elbows, knees and scalp, itching","immune-driven skin cell overproduction, genetics, triggers","topical corticosteroid or vitamin D cream, as directed","moisturising, phototherapy, biologics for severe cases"),
 ("Eczema","dry itchy red patches, cracking, weeping skin","skin barrier defect, allergy, genetics, irritants","topical corticosteroid or calcineurin inhibitor during flares","daily emollients, avoiding triggers, lukewarm bathing"),
 ("Conjunctivitis","red itchy eye, watery or sticky discharge, gritty feeling","viral or bacterial infection, or allergy","antibiotic eye drops for bacterial cases, antihistamine drops for allergy","cool compresses, artificial tears, good hand hygiene"),
 ("Otitis Media","ear pain, fever, irritability, poor sleep","Eustachian tube blockage after a cold, bacterial infection","paracetamol or ibuprofen for pain, amoxicillin if severe","pain relief, follow-up if symptoms persist"),
 ("Celiac Disease","diarrhoea, bloating, weight loss, anaemia","autoimmune reaction to gluten in the small intestine","no drug treatment; strict gluten-free diet","lifelong gluten avoidance, nutritional monitoring"),
 ("Irritable Bowel Syndrome","abdominal cramps, bloating, alternating diarrhoea and constipation","gut-brain interaction, stress, food triggers","antispasmodic or fibre supplements, as prescribed","low-FODMAP diet under guidance, stress management, regular exercise"),
 ("Anxiety Disorder","excessive worry, restlessness, racing heart, poor concentration","genetics, stress, personality, substance use","SSRI such as escitalopram (generic), as prescribed","cognitive behavioural therapy, relaxation techniques, limiting caffeine"),
 ("Bronchitis","persistent cough with mucus, chest discomfort, mild fever","viral infection, smoking, air pollution","salbutamol inhaler if wheezing, as directed","rest, fluids, stopping smoking, humidified air"),
 ("Sinusitis","facial pressure, blocked nose, thick discharge, reduced smell","viral infection, allergies, nasal polyps","nasal corticosteroid spray, saline rinses","steam inhalation, adequate fluids, avoiding allergens"),
 ("Tonsillitis","sore throat, painful swallowing, fever, swollen tonsils","viral or streptococcal infection","penicillin or amoxicillin if bacterial, as prescribed","rest, warm fluids, throat lozenges, pain relief"),
 ("Common Cold","runny nose, sneezing, sore throat, mild cough","rhinovirus and other respiratory viruses","paracetamol or antihistamine for symptom relief","rest, fluids, hand hygiene"),
 ("Measles","high fever, cough, runny nose, red eyes, rash starting on the face","measles virus spread by respiratory droplets","no antiviral; vitamin A and paracetamol as advised","isolation, fluids, MMR vaccination"),
 ("Mumps","swollen painful salivary glands, fever, headache","mumps virus spread by saliva","no antiviral; paracetamol for pain, as directed","rest, fluids, MMR vaccination"),
 ("Rubella","mild fever, pink rash, swollen lymph nodes","rubella virus spread by droplets","no antiviral; paracetamol for fever and aches","rest, MMR vaccination, avoiding exposure in pregnancy"),
 ("Hepatitis A","nausea, jaundice, fatigue, dark urine","hepatitis A virus from contaminated food and water","no specific drug; supportive care","rest, hydration, hepatitis A vaccination, hygiene"),
 ("Hepatitis C","often none; fatigue, jaundice, liver tenderness","hepatitis C virus spread by blood exposure","sofosbuvir-based direct-acting antivirals, as prescribed","liver monitoring, avoiding alcohol, completing treatment"),
 ("HIV Infection","flu-like illness early, later weight loss and infections","HIV spread by blood and sexual contact","antiretroviral therapy such as tenofovir-based regimens, as prescribed","lifelong treatment, regular viral load tests, safer sex"),
 ("Herpes Zoster (Shingles)","painful blistering rash on one side of the body, burning pain","reactivated varicella-zoster virus","valacyclovir (generic) if started early, as prescribed","pain relief, cool compresses, shingles vaccination"),
 ("Cold Sores","tingling then small blisters around the lips","herpes simplex virus type 1","aciclovir cream (generic), as directed","avoiding touching sores, sun protection, hand hygiene"),
 ("Athlete's Foot","itching and scaling between the toes, cracked skin","fungal infection in warm moist shoes","clotrimazole cream (generic), as directed","keeping feet dry, changing socks, breathable shoes"),
 ("Ringworm","circular red scaly patch with a clear centre, itching","dermatophyte fungi spread by contact with people or animals","terbinafine or clotrimazole cream, as directed","keeping skin dry, not sharing towels, treating pets"),
 ("Acne Vulgaris","pimples, blackheads, whiteheads, oily skin","clogged pores, hormones, bacteria","benzoyl peroxide or topical retinoid (generic), as directed","gentle cleansing, non-comedogenic products, avoiding picking"),
 ("Rosacea","facial redness, flushing, visible blood vessels, small bumps","unclear; genetics, triggers such as heat and spicy food","metronidazole or ivermectin cream (generic), as directed","sun protection, avoiding triggers, gentle skincare"),
 ("Urticaria (Hives)","itchy raised welts that come and go","allergic reactions, infections, stress, unknown triggers","antihistamine such as cetirizine (generic), as directed","avoiding triggers, cool compresses, urgent help for face or throat swelling"),
 ("Allergic Rhinitis","sneezing, itchy nose and eyes, runny nose","pollen, dust mites, pet dander","antihistamine or nasal steroid spray, as directed","allergen avoidance, saline rinses, air filters"),
 ("Peptic Ulcer Disease","burning stomach pain, bloating, nausea","H. pylori infection, NSAID use","proton pump inhibitor with antibiotics for H. pylori, as prescribed","avoiding NSAIDs, alcohol and smoking, completing treatment"),
 ("Appendicitis","pain starting near the navel and moving to the lower right, fever, vomiting","blockage of the appendix by faeces or infection","surgical removal (appendectomy) with antibiotics","urgent surgical assessment, rest, fluids before surgery"),
 ("Gallstones","pain in the upper right abdomen after fatty meals, nausea","cholesterol or pigment stones in the gallbladder","ursodeoxycholic acid (generic) in selected cases, as prescribed","low-fat diet, weight management, surgery if symptomatic"),
 ("Pancreatitis","severe upper abdominal pain radiating to the back, vomiting","gallstones, alcohol, high triglycerides","pain relief and fluids in hospital, as prescribed","fasting initially, treating the cause, avoiding alcohol"),
 ("Diverticulitis","left lower abdominal pain, fever, constipation or diarrhoea","infection or inflammation of diverticula in the colon","amoxicillin-clavulanate or metronidazole, as prescribed","liquid diet during flares, high-fibre diet afterwards"),
 ("Constipation","infrequent or hard stools, straining, bloating","low fibre, dehydration, inactivity, some medicines","polyethylene glycol (generic) laxative, as directed","more fibre and fluids, regular physical activity, toilet routine"),
 ("Infectious Diarrhoea","watery stools, cramps, nausea, low-grade fever","viruses or bacteria from contaminated food or water","oral rehydration salts; antibiotics only in selected cases","hydration, hand hygiene, safe food and water"),
 ("Crohn's Disease","diarrhoea, abdominal pain, weight loss, fatigue","immune-mediated inflammation of the gut, genetics","mesalazine or biologics such as infliximab, as prescribed","regular specialist review, nutritional support, smoking cessation"),
 ("Ulcerative Colitis","bloody diarrhoea, urgency, abdominal cramps","immune-mediated inflammation of the colon lining","mesalazine (generic) or corticosteroids during flares, as prescribed","specialist monitoring, diet support, surveillance colonoscopy"),
 ("Osteoporosis","often none; fractures from minor falls, back pain, height loss","ageing, low oestrogen, low calcium and vitamin D","bisphosphonate such as alendronate (generic), as prescribed","calcium and vitamin D intake, weight-bearing exercise, fall prevention"),
 ("Sciatica","shooting pain from the back into the leg, numbness, tingling","compressed or irritated sciatic nerve, often from a disc problem","NSAID or gabapentin for nerve pain, as prescribed","physiotherapy, staying active, heat or cold packs"),
 ("Low Back Pain","ache or stiffness in the lower back, limited movement","muscle strain, poor posture, disc problems","paracetamol or NSAID for short-term pain, as directed","staying active, physiotherapy, ergonomic posture"),
 ("Fibromyalgia","widespread pain, fatigue, poor sleep, memory difficulty","unclear; central pain processing, genetics, stress","duloxetine or pregabalin (generic), as prescribed","exercise, sleep hygiene, cognitive behavioural therapy"),
 ("Systemic Lupus Erythematosus","joint pain, facial rash, fatigue, fever, mouth ulcers","autoimmune reaction, genetics, sunlight and infections as triggers","hydroxychloroquine (generic) as background therapy","sun protection, regular kidney and blood tests, specialist care"),
 ("Multiple Sclerosis","numbness, vision problems, weakness, balance issues","autoimmune damage to the myelin covering nerves","disease-modifying therapy such as interferon or ocrelizumab, as prescribed","physiotherapy, relapse management, vitamin D monitoring"),
 ("Parkinson's Disease","tremor at rest, slowness of movement, stiffness","loss of dopamine-producing neurons, age, genetics","levodopa with carbidopa (generic), as prescribed","physiotherapy, speech therapy, balanced diet"),
 ("Alzheimer's Disease","memory loss, confusion, difficulty with daily tasks","protein build-up in the brain, age, genetics","donepezil (generic) for symptom support, as prescribed","cognitive activity, social engagement, caregiver support"),
 ("Bipolar Disorder","episodes of elevated mood and depression, changes in energy","genetics, brain chemistry, stress","mood stabiliser such as lithium or valproate (generic), as prescribed","regular sleep, medication adherence, psychological therapy"),
 ("Schizophrenia","hallucinations, delusions, disorganised thinking, withdrawal","genetic and environmental factors, brain chemistry","antipsychotic such as risperidone (generic), as prescribed","psychosocial support, rehabilitation, regular follow-up"),
 ("Insomnia","difficulty falling or staying asleep, daytime tiredness","stress, poor sleep habits, anxiety, medical conditions","short-term sleep medicine if needed, as prescribed","cognitive behavioural therapy for insomnia, regular sleep schedule"),
 ("Obstructive Sleep Apnoea","loud snoring, breathing pauses, daytime sleepiness","airway collapse during sleep, obesity, large tonsils","no drug treatment; CPAP device when indicated","weight loss, sleeping on the side, avoiding alcohol before bed"),
 ("Heart Failure","breathlessness, leg swelling, fatigue, reduced exercise tolerance","coronary disease, hypertension, valve problems","ACE inhibitor and beta-blocker (generic), as prescribed","salt and fluid restriction, daily weight checks, exercise rehabilitation"),
 ("Atrial Fibrillation","irregular heartbeat, palpitations, breathlessness","ageing, hypertension, heart disease, thyroid problems","anticoagulant such as apixaban (generic) and rate control drugs, as prescribed","stroke prevention, limiting alcohol and caffeine, regular review"),
 ("Deep Vein Thrombosis","leg swelling, pain, warmth and redness in one leg","immobility, surgery, clotting disorders","anticoagulant such as rivaroxaban (generic), as prescribed","early mobilisation, compression stockings, follow-up"),
 ("Vitamin B12 Deficiency","tiredness, numbness, tingling, memory problems, pale skin","poor absorption, pernicious anaemia, vegan diet","vitamin B12 injections or oral tablets, as prescribed","dietary sources such as eggs and dairy, periodic blood tests"),
 ("Hyperthyroidism","weight loss, rapid heartbeat, tremor, heat intolerance","Graves disease, toxic nodules, thyroiditis","methimazole (generic), as prescribed","beta-blocker for symptom control, regular thyroid tests"),
 ("Polycystic Ovary Syndrome","irregular periods, acne, excess hair growth, weight gain","hormonal imbalance, insulin resistance, genetics","combined oral contraceptive or metformin (generic), as prescribed","weight management, regular exercise, fertility advice when needed"),
 ("Endometriosis","painful periods, pelvic pain, pain during intercourse","tissue like the uterine lining growing outside the uterus","NSAIDs or hormonal contraception, as prescribed","pain management, physiotherapy, specialist surgical review if needed"),
 ("Urinary Incontinence","leaking urine on coughing, sneezing or urgency","weak pelvic floor muscles, childbirth, ageing, obesity","anticholinergic or mirabegron (generic) for urgency, as prescribed","pelvic floor exercises, bladder training, weight loss"),
 ("Benign Prostatic Hyperplasia","weak urine stream, frequent night urination, incomplete emptying","enlargement of the prostate gland with age","alpha blocker such as tamsulosin (generic), as prescribed","regular review, limiting caffeine and night-time fluids"),
 ("Erectile Dysfunction","difficulty achieving or maintaining an erection","vascular disease, diabetes, stress, medications","phosphodiesterase inhibitor such as sildenafil (generic), as prescribed","managing cardiovascular risk, exercise, counselling"),
 ("Gestational Diabetes","often none; excessive thirst, frequent urination in later pregnancy","hormonal changes in pregnancy causing insulin resistance","insulin or metformin (generic) if diet fails, as prescribed","healthy diet, regular activity, blood glucose monitoring"),
 ("Preeclampsia","high blood pressure in pregnancy, headache, swelling, visual changes","placental problems, first pregnancy, pre-existing hypertension","labetalol or nifedipine (generic) for blood pressure, as prescribed","urgent obstetric review, delivery planning, close monitoring"),
 ("Vitamin D Deficiency","bone pain, muscle weakness, fatigue","low sun exposure, poor diet, darker skin, obesity","vitamin D supplements such as cholecalciferol (generic), as prescribed","safe sun exposure, vitamin D-rich foods, follow-up blood tests"),
 ("Obesity","excess body fat, breathlessness on exertion, joint pain","high energy intake, inactivity, genetics, sleep problems","weight-management medicine such as semaglutide, as prescribed","calorie-controlled diet, physical activity, behavioural support"),
 ("Metabolic Syndrome","large waist, raised blood pressure, high blood sugar","insulin resistance, obesity, inactivity","statin or antihypertensive as needed, as prescribed","weight loss, exercise, healthy diet, regular screening"),
 ("Hyperlipidaemia","often none; yellow fatty deposits in some cases","high LDL cholesterol, genetics, poor diet","atorvastatin (generic), as prescribed","low saturated fat diet, exercise, weight control"),
 ("Chronic Obstructive Pulmonary Disease","chronic cough, sputum, breathlessness on exertion","long-term smoking, air pollution, occupational dust","tiotropium (generic) bronchodilator inhaler, as prescribed","stopping smoking, pulmonary rehabilitation, vaccinations"),
 ("Pulmonary Embolism","sudden breathlessness, chest pain, rapid heartbeat, coughing blood","blood clot travelling from the legs to the lungs","anticoagulant such as heparin or apixaban (generic), as prescribed","emergency care, mobilisation, long-term anticoagulation review"),
 ("Pharyngitis","sore throat, painful swallowing, mild fever","viral infection or group A streptococcus","penicillin or amoxicillin if strep confirmed, as prescribed","fluids, salt-water gargles, pain relief"),
 ("Lyme Disease","expanding bull's-eye rash, fever, fatigue, joint pain","bacteria Borrelia burgdorferi spread by tick bites","doxycycline (generic) for early infection, as prescribed","tick checks, prompt review after a tick bite with rash"),
 ("Leprosy","pale or red skin patches with reduced sensation","Mycobacterium leprae spread by prolonged close contact","multidrug therapy with rifampicin, dapsone and clofazimine, as prescribed","completing treatment, checking nerves, preventing disability"),
 ("Cholera","profuse watery diarrhoea, vomiting, rapid dehydration","Vibrio cholerae from contaminated water","oral rehydration solution; antibiotics such as azithromycin in severe cases","rehydration, safe water, sanitation"),
 ("Chikungunya","sudden fever, severe joint pain, rash","chikungunya virus transmitted by Aedes mosquitoes","paracetamol for fever and pain; avoid aspirin","rest, fluids, joint mobilisation, mosquito control"),
 ("Yellow Fever","fever, headache, jaundice, bleeding in severe cases","yellow fever virus transmitted by mosquitoes","no antiviral; supportive care in hospital","yellow fever vaccination, mosquito protection, hydration"),
 ("Rabies","fever, anxiety, hydrophobia, paralysis after an animal bite","rabies virus transmitted through animal bites or scratches","post-exposure vaccine and immunoglobulin, as prescribed","immediate wound washing, urgent medical care after any animal bite"),
 ("Tetanus","jaw stiffness, muscle spasms, difficulty swallowing","Clostridium tetani bacteria entering through wounds","metronidazole and tetanus immunoglobulin, as prescribed","wound cleaning, tetanus booster vaccination, hospital care"),
 ("Meningitis","severe headache, fever, neck stiffness, light sensitivity","bacterial or viral infection of the membranes around the brain","ceftriaxone (generic) given urgently, as prescribed","emergency hospital care, vaccination against meningococcus and pneumococcus"),
]
KEYS = ['name', 'symptoms', 'causes', 'prescription', 'treatment']
DATA = [dict(zip(KEYS, row)) for row in DISEASES]
print('diseases:', len(DATA))

TEMPLATES = [
    'What are the symptoms of {n}?',
    'What causes {n}?',
    'How is {n} prescribed and treated?',
    'Tell me about {n}.',
]

def paragraph(d):
    # The RAG document for one disease
    return (d['name'] + ' is a condition with symptoms including ' + d['symptoms'] +
            '. It is caused by ' + d['causes'] + '. Treatment usually includes ' +
            d['prescription'] + ', along with ' + d['treatment'] + '.')

def answer_text(d):
    # The target format the fine-tuned model must learn
    return ('**Disease:** ' + d['name'] + '\n'
            '**Symptoms:** ' + d['symptoms'] + '.\n'
            '**Causes:** ' + d['causes'] + '.\n'
            '**Prescription:** ' + d['prescription'] + '.\n'
            '**Treatment:** ' + d['treatment'] + '.\n\n'
            '*Synthetic educational data, not medical advice.*')

print(paragraph(DATA[0]))
print()
print(answer_text(DATA[0]))

diseases: 100
Type 2 Diabetes is a condition with symptoms including increased thirst, frequent urination, fatigue, blurred vision. It is caused by insulin resistance, excess body weight, physical inactivity. Treatment usually includes metformin (generic), dose set by a doctor, along with balanced diet, regular exercise, blood sugar monitoring.

**Disease:** Type 2 Diabetes
**Symptoms:** increased thirst, frequent urination, fatigue, blurred vision.
**Causes:** insulin resistance, excess body weight, physical inactivity.
**Prescription:** metformin (generic), dose set by a doctor.
**Treatment:** balanced diet, regular exercise, blood sugar monitoring.

*Synthetic educational data, not medical advice.*


In [4]:

# Load the RAG knowledge base from the Word document.
# Upload disease_knowledge_base_100.docx to the Colab file panel first.
# If the file is missing, we fall back to the generated paragraphs so the notebook still runs.
import os
from docx import Document

DOC_PATH = 'disease_knowledge_base_100.docx'
DOC_TEXT = {}
if os.path.exists(DOC_PATH):
    cur = None
    for p in Document(DOC_PATH).paragraphs:
        if p.style.name.startswith('Heading 2'):
            cur = p.text.split('. ', 1)[1].strip()      # "12. Malaria" -> "Malaria"
        elif cur and p.text.strip():
            DOC_TEXT[cur] = p.text.strip()
            cur = None
    print('loaded from', DOC_PATH)
else:
    DOC_TEXT = {d['name']: paragraph(d) for d in DATA}
    print('WARNING: docx not found, using generated paragraphs')

NAMES = [d['name'] for d in DATA]
missing = [n for n in NAMES if n not in DOC_TEXT]
assert not missing, f'missing from document: {missing}'
PARAS = [DOC_TEXT[n] for n in NAMES]
print('RAG documents:', len(PARAS))
print(PARAS[0][:200])

RAG documents: 100
Type 2 Diabetes is a condition with symptoms including increased thirst, frequent urination, fatigue, blurred vision. It is caused by insulin resistance, excess body weight, physical inactivity. Treat


## 2. Load the base model and get a baseline answer

No formatting instructions are given. Any structure in the output comes from the model itself.

In [5]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(device)

def make_user(question, context=None):
    # ONE prompt builder for training AND inference, so the formats always match
    if context is None:
        return question
    return 'Context:\n' + context + '\n\nQuestion: ' + question + '\nAnswer using only the context.'

def generate(m, question, context=None):
    user = make_user(question, context)
    text = tok.apply_chat_template([{'role': 'user', 'content': user}], tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors='pt').to(device)
    m.eval()
    with torch.no_grad():
        out = m.generate(**enc, max_new_tokens=200, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print('--- BASE model answer ---')
print(generate(model, 'Tell me about ' + DATA[0]['name'] + '.'))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

--- BASE model answer ---
Type 2 diabetes is a chronic metabolic disorder characterized by high blood sugar levels due to insulin resistance or deficiency in the body. It affects approximately 1 in 3 adults worldwide and can lead to serious health complications if left untreated.

The primary symptoms of type 2 diabetes include frequent urination (polyuria), thirst (polydipsia), fatigue, weight loss, and increased hunger. These symptoms often occur without any noticeable physical signs, making it challenging for individuals with type 2 diabetes to recognize their condition.

Diagnosis typically involves a combination of medical history, physical examination, and sometimes blood tests. The diagnosis is made based on the presence of one or more of these symptoms over time.

Treatment options for type 2 diabetes include lifestyle changes such as diet and exercise, medication, and in some cases, surgery. Lifestyle modifications include reducing calorie intake, increasing physical activity,

## 3. Few-shot LoRA fine-tuning

Only **10 diseases** are used, with 2 questions each, in two prompt styles:
- plain questions (no context)
- RAG-style prompts, where the context is that disease's paragraph

That gives 40 training examples in total. The model learns the 5-section format, and it learns to copy facts from the context. Its facts are never memorised from the other 90 diseases.

In [6]:
!pip uninstall -y torchao

FEW_N = 10
TRAIN_D = DATA[:FEW_N]
TRAIN_Q = [t for t in TEMPLATES]        # all 4 phrasings, not 2

def build(q, a):
    prompt = tok.apply_chat_template([{'role': 'user', 'content': q}], tokenize=False, add_generation_prompt=True)
    p_ids = tok(prompt, add_special_tokens=False)['input_ids']
    a_ids = tok(a + tok.eos_token, add_special_tokens=False)['input_ids']
    ids = p_ids + a_ids
    return {'input_ids': ids, 'attention_mask': [1] * len(ids), 'labels': [-100] * len(p_ids) + a_ids}

plain_pairs = [(t.format(n=d['name']), answer_text(d)) for d in TRAIN_D for t in TRAIN_Q]
rag_pairs = [(make_user(t.format(n=d['name']), DOC_TEXT[d['name']]), answer_text(d)) for d in TRAIN_D for t in TRAIN_Q]
pairs = plain_pairs + rag_pairs
print('training examples:', len(pairs), '| diseases seen in training:', FEW_N)

train_ds = [build(q, a) for q, a in pairs]

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir='out',
    num_train_epochs=4,
    per_device_train_batch_size=4,
    learning_rate=3e-4,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    seed=0,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True))

t0 = time.time()
trainer.train()
print(f'Training time: {time.time() - t0:.0f}s')
model.save_pretrained('disease-lora-fewshot')

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
training examples: 80 | diseases seen in training: 10
trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


Step,Training Loss
5,1.417945
10,0.777366
15,0.617812
20,0.481493
25,0.322272
30,0.190345
35,0.151892
40,0.114625
45,0.042419
50,0.034772


Training time: 50s


## 4. RAG over the Word document

The RAG knowledge base is `disease_knowledge_base_100.docx`, one paragraph per disease. Each paragraph is embedded once. For each question, the most similar paragraph is retrieved. The retrieval check runs over all 100 diseases before any generation, so you can see whether retrieval or generation is the bottleneck.

In [7]:

emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
PARA_VEC = emb.encode(PARAS, normalize_embeddings=True)

def retrieve_idx(q):
    v = emb.encode([q], normalize_embeddings=True)[0]
    return int(np.argmax(PARA_VEC @ v))

def retrieve(q):
    return PARAS[retrieve_idx(q)]

hits = [NAMES[retrieve_idx(t.format(n=d['name']))] == d['name'] for d in DATA for t in TEMPLATES]
print(f'Retrieval accuracy over 100 diseases x {len(TEMPLATES)} questions: {np.mean(hits):.1%}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieval accuracy over 100 diseases x 4 questions: 100.0%


## 5. Compare the four setups on a sample

Sample: 5 diseases the model was trained on, and 5 it never saw. Each answer is scored on:
- **Structure**: fraction of the 5 required headers present
- **Name correct**: the answer names the right disease
- **Content recall**: fraction of reference items in the answer

In [8]:
HEADS = ['**Disease:**', '**Symptoms:**', '**Causes:**', '**Prescription:**', '**Treatment:**']

def structure_score(a):
    return sum(h in a for h in HEADS) / len(HEADS)

def content_recall(a, d):
    items = [t.strip().lower() for f in ('symptoms', 'causes', 'treatment') for t in d[f].split(', ')]
    a = a.lower()
    return sum(i in a for i in items) / len(items)

def name_hit(a, d):
    return d['name'].lower() in a.lower()

def ans_for(q, ctx, ft):
    if ft:
        return generate(model, q, ctx)
    with model.disable_adapter():   # LoRA off = base model, same weights
        return generate(model, q, ctx)

SAMPLE_IDX = list(range(5)) + list(range(50, 55))
CONFIGS = [('Base', False, False), ('Fine-tuned', True, False),
           ('Base + RAG', False, True), ('Fine-tuned + RAG', True, True)]

rows = []
for name, ft, rag in CONFIGS:
    for i in SAMPLE_IDX:
        d = DATA[i]
        q = 'Tell me about ' + d['name'] + '.'
        ctx = retrieve(q) if rag else None
        a = ans_for(q, ctx, ft)
        rows.append(dict(config=name, disease=d['name'], seen=i < FEW_N, answer=a,
                         structure=structure_score(a), name_ok=name_hit(a, d),
                         recall=content_recall(a, d)))
    print('done:', name)

df = pd.DataFrame(rows)
df.to_csv('sample_results.csv', index=False)

done: Base
done: Fine-tuned
done: Base + RAG
done: Fine-tuned + RAG


In [9]:
summary = df.groupby('config', sort=False)[['structure', 'name_ok', 'recall']].mean().round(2)
summary.columns = ['Structure', 'Name correct', 'Content recall']
print(summary)
print()
print(df.groupby(['config', 'seen'], sort=False)[['structure', 'name_ok']].mean().round(2))

                  Structure  Name correct  Content recall
config                                                   
Base                    0.0           0.9            0.19
Fine-tuned              1.0           1.0            0.63
Base + RAG              0.0           1.0            0.52
Fine-tuned + RAG        1.0           1.0            1.00

                        structure  name_ok
config           seen                     
Base             True         0.0      1.0
                 False        0.0      0.8
Fine-tuned       True         1.0      1.0
                 False        1.0      1.0
Base + RAG       True         0.0      1.0
                 False        0.0      1.0
Fine-tuned + RAG True         1.0      1.0
                 False        1.0      1.0


## 6. Main test: Fine-tuned + RAG on all 100 diseases

This is the target. For every disease, the model receives the retrieved paragraph and must return the correct disease in the 5-section format. "Passed" means all 5 headers are present and the correct disease is named.

In [10]:
def clean(a):
    # Drop any preamble or echoed context before the first header
    k = a.find('**Disease:**')
    return a[k:] if k >= 0 else a

full = []
for i, d in enumerate(DATA):
    q = 'Tell me about ' + d['name'] + '.'
    idx = retrieve_idx(q)
    raw = generate(model, q, PARAS[idx])
    a = clean(raw)
    s_raw, s = structure_score(raw), structure_score(a)
    full.append(dict(disease=d['name'], seen=i < FEW_N, retrieved_ok=(NAMES[idx] == d['name']),
                     structure_raw=s_raw, structure_clean=s,
                     name_ok=name_hit(a, d),
                     passed_raw=(s_raw == 1.0 and name_hit(raw, d)),
                     passed_clean=(s == 1.0 and name_hit(a, d)),
                     answer_raw=raw, answer_clean=a))
full_df = pd.DataFrame(full)
full_df.to_csv('ft_rag_all100.csv', index=False)

print(f"Passed, raw output       (all 100): {full_df.passed_raw.mean():.0%}")
print(f"Passed, after cleaning   (all 100): {full_df.passed_clean.mean():.0%}")
print(f"  trained diseases ({FEW_N}):  {full_df[full_df.seen].passed_clean.mean():.0%}")
print(f"  unseen diseases ({100 - FEW_N}): {full_df[~full_df.seen].passed_clean.mean():.0%}")
print(f"Retrieval correct: {full_df.retrieved_ok.mean():.0%}")

Passed, raw output       (all 100): 97%
Passed, after cleaning   (all 100): 97%
  trained diseases (10):  100%
  unseen diseases (90): 97%
Retrieval correct: 100%


In [11]:
failed = full_df[~full_df.passed_clean][['disease', 'seen', 'retrieved_ok', 'structure_clean', 'name_ok']]
print('Failures after cleaning:', len(failed))
failed.head(20)

Failures after cleaning: 3


,disease,seen,retrieved_ok,structure_clean,name_ok
26,Cataract,False,True,0.8,True
72,Obstructive Sleep Apnoea,False,True,0.8,True
83,Gestational Diabetes,False,True,0.8,True


In [14]:
ex = full_df.iloc[77]
print('QUESTION: Tell me about', ex.disease, '| trained:', ex.seen)
print(ex.answer_clean)

QUESTION: Tell me about Hyperthyroidism | trained: False
**Disease:** Hyperthyroidism
**Symptoms:** weight loss, rapid heartbeat, tremor, heat intolerance.
**Causes:** Graves disease, toxic nodules, thyroiditis.
**Prescription:** methimazole (generic), as prescribed.
**Treatment:** beta-blocker for symptom control, regular thyroid tests.

*Synthetic educational data, not medical advice.*


## What to observe

- **Structure vs Base:** the fine-tuned model should produce the 5-section format even for diseases it never trained on. That is the few-shot lesson.
- **Retrieval vs generation:** if "Retrieval correct" is high but "Passed" is low, the failure is in generation. If retrieval is low, the failure is in the RAG step, and the fix is better embeddings or query handling.
- **Trained vs unseen:** unseen diseases only pass if the model copies facts from the retrieved paragraph and does not rely on memorised training answers. A large gap between the two is a sign of over-memorisation, which you can reduce by lowering epochs or adding more diverse RAG-style examples.
- **Ideas to extend:** try 3 to 5 training diseases, or a stronger retriever such as bge-small-en-v1.5.